# Borewell Sentinel India
## Phase 2 — Dataset Investigation & Audit
### Task 2.1 — Dataset Structure Analysis

Objective:
Load all 31 extracted datasets and investigate
their dimensions, column names, data types,
and structural consistency.

In [1]:

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

EXTRACTED_DIR = PROJECT_ROOT / "data" / "extracted"

csv_files = sorted(EXTRACTED_DIR.rglob("*.csv"))

datasets = {}

for file in csv_files:
    df = pd.read_csv(
        file,
        dtype=str,
        keep_default_na=False,
        na_filter=False
    )

    datasets[file.stem] = df

print("DATASET LOADING SUMMARY")
print("=" * 55)

print("Files loaded:", len(datasets))
print("Total records:", sum(len(df) for df in datasets.values()))

print("\nAll datasets loaded successfully.")


DATASET LOADING SUMMARY
Files loaded: 31
Total records: 174635

All datasets loaded successfully.


In [2]:

dataset_summary = pd.DataFrame([
    {
        "Dataset": name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Memory_MB": round(
            df.memory_usage(deep=True).sum() / (1024**2), 3
        )
    }
    for name, df in datasets.items()
])

display(dataset_summary)

print("\nTOTAL ROWS:", dataset_summary["Rows"].sum())
print("TOTAL COLUMNS PER FILE:",
      dataset_summary["Columns"].unique())


,Dataset,Rows,Columns,Memory_MB
0,gwq_chemical_parameter_manual_cgwb_an_1961_2025,634,46,1.468
1,gwq_chemical_parameter_manual_cgwb_ap_1961_2025,10950,46,25.077
2,gwq_chemical_parameter_manual_cgwb_ar_1961_2025,104,46,0.238
3,gwq_chemical_parameter_manual_cgwb_as_1961_2025,1874,46,4.272
4,gwq_chemical_parameter_manual_cgwb_br_1961_2025,3112,46,7.072
5,gwq_chemical_parameter_manual_cgwb_cg_1961_2025,10350,46,23.628
6,gwq_chemical_parameter_manual_cgwb_ch_1961_2025,36,46,0.082
7,gwq_chemical_parameter_manual_cgwb_dl_1961_2025,1169,46,2.677
8,gwq_chemical_parameter_manual_cgwb_dn_1961_2025,169,46,0.395
9,gwq_chemical_parameter_manual_cgwb_ga_1961_2025,674,46,1.527



TOTAL ROWS: 174635
TOTAL COLUMNS PER FILE: [46]


In [3]:

reference_columns = list(next(iter(datasets.values())).columns)

schema_results = []

for name, df in datasets.items():

    current_columns = list(df.columns)

    schema_results.append({
        "Dataset": name,
        "Same_Column_Count": len(current_columns) == len(reference_columns),
        "Same_Column_Names": set(current_columns) == set(reference_columns),
        "Same_Column_Order": current_columns == reference_columns,
        "Column_Count": len(current_columns)
    })

schema_df = pd.DataFrame(schema_results)

display(schema_df)

print("\nSchema consistency:")
print(schema_df.drop(columns="Dataset").all())


,Dataset,Same_Column_Count,Same_Column_Names,Same_Column_Order,Column_Count
0,gwq_chemical_parameter_manual_cgwb_an_1961_2025,True,True,True,46
1,gwq_chemical_parameter_manual_cgwb_ap_1961_2025,True,True,True,46
2,gwq_chemical_parameter_manual_cgwb_ar_1961_2025,True,True,True,46
3,gwq_chemical_parameter_manual_cgwb_as_1961_2025,True,True,True,46
4,gwq_chemical_parameter_manual_cgwb_br_1961_2025,True,True,True,46
5,gwq_chemical_parameter_manual_cgwb_cg_1961_2025,True,True,True,46
6,gwq_chemical_parameter_manual_cgwb_ch_1961_2025,True,True,True,46
7,gwq_chemical_parameter_manual_cgwb_dl_1961_2025,True,True,True,46
8,gwq_chemical_parameter_manual_cgwb_dn_1961_2025,True,True,True,46
9,gwq_chemical_parameter_manual_cgwb_ga_1961_2025,True,True,True,46



Schema consistency:
Same_Column_Count    True
Same_Column_Names    True
Same_Column_Order    True
Column_Count         True
dtype: bool


In [4]:

print("REFERENCE DATASET COLUMNS")
print("=" * 55)

for index, column in enumerate(reference_columns, start=1):
    print(f"{index:02d}. {column}")

print("\nDATA TYPES")
display(
    next(iter(datasets.values())).dtypes
    .reset_index()
    .rename(columns={"index": "Column", 0: "Data Type"})
)


REFERENCE DATASET COLUMNS
01. SlNo
02. Station
03. Agency
04. State LGD Code
05. State
06. District LGD Code
07. District
08. Tehsil
09. Block
10. Village
11. River
12. Basin
13. Tributary
14. Subtributary
15. SubSubtributary
16. Local River
17. Latitude
18. Longitude
19. Data Acquisition Time
20. Potential of Hydrogen (pH)
21. Electric Conductivity (μS/cm)
22. Total Dissolved Solids (mg/L)
23. Carbonate (mg/L)
24. Bicarbonate (mg/L)
25. Total Alkalinity (mg/L as CaCO3)
26. Chloride (mg/L)
27. Nitrate N (mgN/L)
28. Sulphate (mg/L)
29. Phosphate(mg/L)
30. Silica(mg/L)
31. Fluoride (mg/L)
32. Total Hardness (mgCaCO3/L)
33. Calcium (mg/L)
34. Magnesium (mg/L)
35. Sodium (mg/L)
36. Potassium (mg/L)
37. Iron(mg/L)
38. Arsenic (mg/L)
39. Uranium(mg/L)
40. Manganese (mg/L)
41. Copper (mg/L)
42. Lead (mg/L)
43. Zinc (mg/L)
44. Nickel (mg/L)
45. Cadmium (mg/L)
46. Chromium (mg/L)

DATA TYPES


,Column,Data Type
0,SlNo,str
1,Station,str
2,Agency,str
3,State LGD Code,str
4,State,str
5,District LGD Code,str
6,District,str
7,Tehsil,str
8,Block,str
9,Village,str


In [5]:
from collections import Counter

# Candidate missing-value markers.
# These will be counted, not automatically replaced.
MISSING_TOKENS = {
    "NA",
    "N/A",
    "NULL",
    "NONE",
    "NAN",
    "-",
    "--",
    "NOT AVAILABLE",
    "NOT REPORTED",
    "NOT APPLICABLE"
}

missing_audit = []

for column in reference_columns:

    blank_count = 0
    placeholder_counts = Counter()

    for df in datasets.values():

        values = df[column].astype(str).str.strip()

        # Count empty and whitespace-only cells
        blank_count += values.eq("").sum()

        # Count possible missing-value markers
        candidates = values[
            values.str.upper().isin(MISSING_TOKENS)
        ]

        placeholder_counts.update(
            candidates.str.upper().value_counts().to_dict()
        )

    missing_audit.append({
        "Column": column,
        "Blank_Count": blank_count,
        "Placeholder_Count": sum(placeholder_counts.values()),
        "Placeholder_Values": dict(placeholder_counts)
    })

missing_audit_df = pd.DataFrame(missing_audit)

display(
    missing_audit_df.sort_values(
        by=["Blank_Count", "Placeholder_Count"],
        ascending=False
    ).reset_index(drop=True)
)

print("TOTAL BLANK CELLS:",
      missing_audit_df["Blank_Count"].sum())

print("TOTAL CANDIDATE PLACEHOLDERS:",
      missing_audit_df["Placeholder_Count"].sum())

,Column,Blank_Count,Placeholder_Count,Placeholder_Values
0,Nitrate N (mgN/L),174635,0,{}
1,Fluoride (mg/L),174635,0,{}
2,Potassium (mg/L),174635,0,{}
3,Lead (mg/L),173841,0,{}
4,Copper (mg/L),173839,0,{}
5,Zinc (mg/L),173839,0,{}
6,Manganese (mg/L),173838,0,{}
7,Nickel (mg/L),173838,0,{}
8,Cadmium (mg/L),173838,0,{}
9,Chromium (mg/L),173838,0,{}


TOTAL BLANK CELLS: 3032074
TOTAL CANDIDATE PLACEHOLDERS: 1056236


In [6]:
from collections import Counter

# Coordinates and all chemical measurement columns
numeric_columns = (
    ["Latitude", "Longitude"]
    + reference_columns[19:]
)

numeric_audit = []

for column in numeric_columns:

    nonblank_count = 0
    invalid_values = Counter()

    for df in datasets.values():

        values = df[column].astype(str).str.strip()

        # Exclude blank cells
        values = values[values != ""]

        nonblank_count += len(values)

        # Identify values that cannot be parsed as numbers
        converted = pd.to_numeric(values, errors="coerce")

        invalid = values[converted.isna()]

        invalid_values.update(invalid.value_counts().to_dict())

    failure_count = sum(invalid_values.values())

    numeric_audit.append({
        "Column": column,
        "Nonblank_Values": nonblank_count,
        "Invalid_Numeric_Count": failure_count,
        "Invalid_Rate_%": round(
            (failure_count / nonblank_count) * 100, 3
        ) if nonblank_count else 0,
        "Invalid_Examples": dict(invalid_values.most_common(10))
    })

numeric_audit_df = pd.DataFrame(numeric_audit)

display(
    numeric_audit_df.sort_values(
        by="Invalid_Numeric_Count",
        ascending=False
    ).reset_index(drop=True)
)

print("TOTAL INVALID NUMERIC VALUES:",
      numeric_audit_df["Invalid_Numeric_Count"].sum())

,Column,Nonblank_Values,Invalid_Numeric_Count,Invalid_Rate_%,Invalid_Examples
0,Potential of Hydrogen (pH),162733,4,0.002,"{'7.147.44': 1, '7.47.75': 1, '7.277.4': 1, '7..."
1,Total Alkalinity (mg/L as CaCO3),79937,1,0.001,{'290.2575.4': 1}
2,Iron(mg/L),38703,1,0.003,{'0.030.05': 1}
3,Electric Conductivity (μS/cm),173029,0,0.000,{}
4,Latitude,174483,0,0.000,{}
5,Total Dissolved Solids (mg/L),30151,0,0.000,{}
6,Carbonate (mg/L),120229,0,0.000,{}
7,Bicarbonate (mg/L),147171,0,0.000,{}
8,Chloride (mg/L),162034,0,0.000,{}
9,Nitrate N (mgN/L),0,0,0.000,{}


TOTAL INVALID NUMERIC VALUES: 6


In [7]:
invalid_records = []

for dataset_name, df in datasets.items():

    for column in numeric_columns:

        raw_values = df[column].astype(str).str.strip()

        numeric_values = pd.to_numeric(
            raw_values,
            errors="coerce"
        )

        invalid_mask = (
            raw_values.ne("")
            & numeric_values.isna()
        )

        for index in df.index[invalid_mask]:

            invalid_records.append({
                "Dataset": dataset_name,
                "Row_Index": index,
                "Station": df.at[index, "Station"],
                "State": df.at[index, "State"],
                "District": df.at[index, "District"],
                "Date": df.at[index, "Data Acquisition Time"],
                "Column": column,
                "Original_Value": df.at[index, column]
            })

invalid_records_df = pd.DataFrame(invalid_records)

pd.set_option("display.max_colwidth", 100)

display(invalid_records_df)

print("TOTAL INVALID RECORDS:", len(invalid_records_df))

,Dataset,Row_Index,Station,State,District,Date,Column,Original_Value
0,gwq_chemical_parameter_manual_cgwb_tn_1961_2025,1360,Dharapuram2,Tamil Nadu,Erode,25-05-2001 00:00,Potential of Hydrogen (pH),7.147.44
1,gwq_chemical_parameter_manual_cgwb_tn_1961_2025,5484,Ramanathapuram2,Tamil Nadu,Ramanathapuram,25-05-2003 00:00,Potential of Hydrogen (pH),7.47.75
2,gwq_chemical_parameter_manual_cgwb_tn_1961_2025,5495,Ramanathapuram2,Tamil Nadu,Ramanathapuram,25-05-2016 00:00,Potential of Hydrogen (pH),7.277.4
3,gwq_chemical_parameter_manual_cgwb_tn_1961_2025,5496,Ramanathapuram2,Tamil Nadu,Ramanathapuram,25-05-2017 00:00,Potential of Hydrogen (pH),7.597.61
4,gwq_chemical_parameter_manual_cgwb_tn_1961_2025,1360,Dharapuram2,Tamil Nadu,Erode,25-05-2001 00:00,Total Alkalinity (mg/L as CaCO3),290.2575.4
5,gwq_chemical_parameter_manual_cgwb_tn_1961_2025,5496,Ramanathapuram2,Tamil Nadu,Ramanathapuram,25-05-2017 00:00,Iron(mg/L),0.030.05


TOTAL INVALID RECORDS: 6


In [8]:
date_audit = []

for dataset_name, df in datasets.items():

    raw_dates = df["Data Acquisition Time"].astype(str).str.strip()

    parsed_dates = pd.to_datetime(
        raw_dates,
        format="mixed",
        dayfirst=True,
        errors="coerce"
    )

    invalid_dates = raw_dates[parsed_dates.isna()]

    date_audit.append({
        "Dataset": dataset_name,
        "Total_Records": len(df),
        "Valid_Dates": parsed_dates.notna().sum(),
        "Invalid_Dates": parsed_dates.isna().sum(),
        "Unique_Dates": parsed_dates.nunique(),
        "Earliest_Date": parsed_dates.min(),
        "Latest_Date": parsed_dates.max(),
        "Invalid_Examples": invalid_dates.unique().tolist()[:10]
    })

date_audit_df = pd.DataFrame(date_audit)

display(date_audit_df)

print("\nDATE AUDIT TOTALS")
print("=" * 50)
print("Total records:", date_audit_df["Total_Records"].sum())
print("Valid dates:", date_audit_df["Valid_Dates"].sum())
print("Invalid dates:", date_audit_df["Invalid_Dates"].sum())

print("\nOVERALL DATE RANGE")
print("Earliest:", date_audit_df["Earliest_Date"].min())
print("Latest:", date_audit_df["Latest_Date"].max())

,Dataset,Total_Records,Valid_Dates,Invalid_Dates,Unique_Dates,Earliest_Date,Latest_Date,Invalid_Examples
0,gwq_chemical_parameter_manual_cgwb_an_1961_2025,634,634,0,13,2004-05-25,2021-05-30 08:00:00,[]
1,gwq_chemical_parameter_manual_cgwb_ap_1961_2025,10950,10950,0,28,2000-05-25,2023-05-25 12:00:00,[]
2,gwq_chemical_parameter_manual_cgwb_ar_1961_2025,104,104,0,15,2005-05-25,2021-05-30 08:00:00,[]
3,gwq_chemical_parameter_manual_cgwb_as_1961_2025,1874,1874,0,18,2000-05-25,2021-05-30 08:00:00,[]
4,gwq_chemical_parameter_manual_cgwb_br_1961_2025,3112,3112,0,18,2000-05-25,2021-05-30 08:00:00,[]
5,gwq_chemical_parameter_manual_cgwb_cg_1961_2025,10350,10350,0,2106,2000-05-25,2024-03-06 14:43:00,[]
6,gwq_chemical_parameter_manual_cgwb_ch_1961_2025,36,36,0,17,2000-05-25,2021-05-30 08:00:00,[]
7,gwq_chemical_parameter_manual_cgwb_dl_1961_2025,1169,1169,0,22,2000-05-25,2021-05-30 08:00:00,[]
8,gwq_chemical_parameter_manual_cgwb_dn_1961_2025,169,169,0,21,2000-05-25,2020-05-30 08:00:00,[]
9,gwq_chemical_parameter_manual_cgwb_ga_1961_2025,674,674,0,54,2000-05-25,2023-07-04 10:29:00,[]



DATE AUDIT TOTALS
Total records: 174635
Valid dates: 174635
Invalid dates: 0

OVERALL DATE RANGE
Earliest: 2000-05-25 00:00:00
Latest: 2025-05-08 10:02:00


In [9]:
combined_df = pd.concat(
    [
        df.assign(_source_file=name)
        for name, df in datasets.items()
    ],
    ignore_index=True
)

combined_df["_parsed_date"] = pd.to_datetime(
    combined_df["Data Acquisition Time"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

combined_df["_date_only"] = combined_df["_parsed_date"].dt.date

print("COMBINED DATASET SUMMARY")
print("=" * 50)
print("Total records:", len(combined_df))
print("Total columns:", len(combined_df.columns))
print("Source files:", combined_df["_source_file"].nunique())
print("Invalid parsed dates:", combined_df["_parsed_date"].isna().sum())
print("Unique states:", combined_df["State"].nunique())
print("Unique districts:", combined_df["District"].nunique())
print("Date range:", combined_df["_parsed_date"].min(),
      "to", combined_df["_parsed_date"].max())

COMBINED DATASET SUMMARY
Total records: 174635
Total columns: 49
Source files: 31
Invalid parsed dates: 0
Unique states: 31
Unique districts: 596
Date range: 2000-05-25 00:00:00 to 2025-05-08 10:02:00


In [10]:
combined_df["_year"] = combined_df["_parsed_date"].dt.year

year_summary = (
    combined_df.groupby("_year")
    .agg(
        Records=("Station", "size"),
        States=("State", "nunique"),
        Stations=("Station", "nunique")
    )
    .reset_index()
)

print("RECORDS BY YEAR")
display(year_summary)

print("RECORDS BY STATE")
state_summary = (
    combined_df.groupby("State")
    .agg(
        Records=("Station", "size"),
        Districts=("District", "nunique"),
        Stations=("Station", "nunique"),
        First_Record=("_parsed_date", "min"),
        Last_Record=("_parsed_date", "max")
    )
    .sort_values("Records", ascending=False)
    .reset_index()
)

display(state_summary)

RECORDS BY YEAR


,_year,Records,States,Stations
0,2000,6645,27,6493
1,2001,5747,23,5622
2,2002,6862,24,6688
3,2003,6486,25,6321
4,2004,7332,24,7169
5,2005,6215,30,6085
6,2006,6022,28,5927
7,2007,6662,28,6531
8,2008,6649,27,6493
9,2009,7440,30,7294


RECORDS BY STATE


,State,Records,Districts,Stations,First_Record,Last_Record
0,Madhya Pradesh,19230,51,1670,2000-05-25,2022-05-31 11:00:00
1,Maharashtra,18249,35,1758,2000-05-25,2024-08-21 10:09:00
2,Odisha,15400,30,1897,2000-05-25,2021-05-30 08:00:00
3,Karnataka,13007,27,2007,2000-05-25,2025-05-08 10:02:00
4,Rajasthan,12350,34,1606,2000-05-25,2024-06-30 09:30:00
5,Andhra Pradesh,10950,14,1538,2000-05-25,2023-05-25 12:00:00
6,Chhattisgarh,10350,23,1255,2000-05-25,2024-03-06 14:43:00
7,Gujarat,9807,28,958,2000-05-25,2021-05-30 08:00:00
8,Kerala,9473,15,1048,2000-05-25,2022-04-10 13:01:00
9,Tamil Nadu,8419,33,1215,2000-05-25,2024-05-30 12:09:00


In [11]:
station_keys = ["Station", "State", "District"]

station_summary = (
    combined_df.groupby(station_keys, dropna=False)
    .agg(
        Records=("Station", "size"),
        Unique_Dates=("_date_only", "nunique"),
        Unique_Latitudes=("Latitude", "nunique"),
        Unique_Longitudes=("Longitude", "nunique"),
        First_Sampled=("_parsed_date", "min"),
        Last_Sampled=("_parsed_date", "max")
    )
    .reset_index()
)

print("STATION IDENTITY SUMMARY")
print("=" * 50)
print("Total provisional station identities:", len(station_summary))
print("Stations with 1 record:",
      (station_summary["Records"] == 1).sum())
print("Stations with 2+ records:",
      (station_summary["Records"] >= 2).sum())
print("Stations with 5+ records:",
      (station_summary["Records"] >= 5).sum())
print("Stations with 10+ records:",
      (station_summary["Records"] >= 10).sum())

print("\nMOST FREQUENTLY SAMPLED STATIONS")
display(
    station_summary.sort_values(
        "Records", ascending=False
    ).head(20)
)

print("\nSTATION NAME COLLISIONS")
station_name_check = (
    combined_df.groupby("Station")
    .agg(
        States=("State", "nunique"),
        Districts=("District", "nunique")
    )
    .reset_index()
)

display(
    station_name_check[
        (station_name_check["States"] > 1)
        | (station_name_check["Districts"] > 1)
    ].sort_values("Districts", ascending=False)
)

STATION IDENTITY SUMMARY
Total provisional station identities: 23856
Stations with 1 record: 3641
Stations with 2+ records: 20215
Stations with 5+ records: 13768
Stations with 10+ records: 7036

MOST FREQUENTLY SAMPLED STATIONS


,Station,State,District,Records,Unique_Dates,Unique_Latitudes,Unique_Longitudes,First_Sampled,Last_Sampled
585,Alampur,Andhra Pradesh,ANANTAPUR,39,22,3,3,2000-05-25,2022-05-31 11:00:00
22671,Tuman,Chhattisgarh,KORBA,28,22,2,2,2000-05-25,2023-12-01 15:57:00
6545,Dongargaon,Maharashtra,Gondia,26,16,2,2,2000-05-25,2015-05-25 00:00:00
4657,Chaparda,Maharashtra,Yavatmal,25,23,1,1,2000-05-25,2023-05-20 12:28:00
3304,Bhainswal,Haryana,SONIPAT,24,24,1,1,2000-05-25,2023-05-18 12:40:00
18868,Ralla,Punjab,MANSA,24,24,1,1,2000-05-25,2023-05-06 09:19:00
6077,Dhanana,Haryana,BHIWANI,24,24,1,1,2000-05-25,2023-05-10 14:24:00
9213,Jagsi,Haryana,SONIPAT,24,24,1,1,2000-05-25,2023-05-18 09:15:00
12374,Kothulna,Maharashtra,Nagpur,23,23,1,1,2000-05-25,2022-05-21 09:52:00
19648,Sabri,Haryana,YAMUNANAGAR,23,23,1,1,2000-05-25,2023-05-18 15:21:00



STATION NAME COLLISIONS


,Station,States,Districts
18211,Rampur,6,11
7639,Govindpur,5,8
18192,Ramnagar,8,8
15399,Narayanpur,7,8
18227,Rampura,6,7
...,...,...,...
3752,Birpur,2,2
22646,Wagholi,1,2
22673,Wandur,2,2
22706,Warud,1,2


In [12]:
# 1. Exact duplicate records across original 46 columns
exact_duplicate_mask = combined_df.duplicated(
    subset=reference_columns,
    keep="first"
)

exact_duplicate_rows = exact_duplicate_mask.sum()

exact_duplicate_groups = combined_df[
    combined_df.duplicated(
        subset=reference_columns,
        keep=False
    )
]

# 2. Repeated station-date combinations
station_date_keys = [
    "Station",
    "State",
    "District",
    "_date_only"
]

station_date_counts = (
    combined_df.groupby(station_date_keys, dropna=False)
    .size()
    .reset_index(name="Records")
)

repeated_station_dates = (
    station_date_counts[
        station_date_counts["Records"] > 1
    ]
    .sort_values("Records", ascending=False)
)

print("DUPLICATE AUDIT")
print("=" * 50)

print("Exact duplicate rows beyond the first occurrence:",
      exact_duplicate_rows)

print("Rows belonging to exact duplicate groups:",
      len(exact_duplicate_groups))

print("Station-date groups with multiple records:",
      len(repeated_station_dates))

print("Records belonging to repeated station-date groups:",
      repeated_station_dates["Records"].sum())

print("Excess observations in repeated station-date groups:",
      (repeated_station_dates["Records"] - 1).sum())

print("\nTOP REPEATED STATION-DATE COMBINATIONS")
display(repeated_station_dates.head(25))

DUPLICATE AUDIT
Exact duplicate rows beyond the first occurrence: 0
Rows belonging to exact duplicate groups: 0
Station-date groups with multiple records: 280
Records belonging to repeated station-date groups: 565
Excess observations in repeated station-date groups: 285

TOP REPEATED STATION-DATE COMBINATIONS


,Station,State,District,_date_only,Records
24795,Bhandi-Shegaon,Maharashtra,Solapur,2023-05-20,3
33515,Chaparda,Maharashtra,Yavatmal,2023-05-20,3
1904,ARNIYALI,Rajasthan,BARMER,2023-06-30,3
64837,Insuli,Maharashtra,Sindudurg,2023-05-20,3
49392,Eluru III,Andhra Pradesh,WEST GODAVARI,2002-05-25,3
4350,Alampur,Andhra Pradesh,ANANTAPUR,2001-05-25,2
174207,Zendewadi-1,Maharashtra,Pune,2023-05-20,2
2038,Aalandh,Maharashtra,Aurangabad,2022-05-21,2
2102,Abad PES polytec,Maharashtra,Aurangabad,2022-05-21,2
2108,Abad PWD RH,Maharashtra,Aurangabad,2022-05-21,2


In [13]:
repeated_records = combined_df[
    combined_df.duplicated(
        subset=station_date_keys,
        keep=False
    )
].copy()

group_details = []

for key, group in repeated_records.groupby(
    station_date_keys,
    dropna=False,
    sort=False
):
    varying_columns = [
        col for col in reference_columns
        if group[col].nunique(dropna=False) > 1
    ]

    group_details.append({
        "_key": key,
        "Records": len(group),
        "Different_Agencies": group["Agency"].nunique(),
        "Different_Source_Files": group["_source_file"].nunique(),
        "Varying_Column_Count": len(varying_columns),
        "Varying_Columns": varying_columns
    })

repeated_group_audit = pd.DataFrame(group_details)

print("REPEATED GROUP ANALYSIS")
print("=" * 55)
print("Groups with differences in at least one original column:",
      (repeated_group_audit["Varying_Column_Count"] > 0).sum())

print("Groups with different agencies:",
      (repeated_group_audit["Different_Agencies"] > 1).sum())

display(
    repeated_group_audit.drop(columns="_key")
    .sort_values(
        ["Varying_Column_Count", "Records"],
        ascending=False
    )
    .head(25)
)

print("\nDETAILS OF FIVE MOST VARIABLE GROUPS")

for _, item in repeated_group_audit.sort_values(
    ["Varying_Column_Count", "Records"],
    ascending=False
).head(5).iterrows():

    key = item["_key"]

    mask = pd.Series(True, index=combined_df.index)

    for column, value in zip(station_date_keys, key):
        mask &= combined_df[column].eq(value)

    group = combined_df.loc[mask]

    detail_columns = list(dict.fromkeys(
        station_date_keys
        + ["_source_file", "SlNo", "Agency",
           "Latitude", "Longitude", "Data Acquisition Time"]
        + item["Varying_Columns"]
    ))

    print("\nStation-date group:", key)
    display(group[detail_columns])

REPEATED GROUP ANALYSIS
Groups with differences in at least one original column: 280
Groups with different agencies: 0


,Records,Different_Agencies,Different_Source_Files,Varying_Column_Count,Varying_Columns
194,2,1,1,18,"[SlNo, Tehsil, Block, Latitude, Longitude, Potential of Hydrogen (pH), Electric Conductivity (μS..."
196,2,1,1,18,"[SlNo, Tehsil, Block, Village, Latitude, Longitude, Potential of Hydrogen (pH), Electric Conduct..."
266,2,1,1,18,"[SlNo, Tehsil, Block, Village, Latitude, Longitude, Potential of Hydrogen (pH), Electric Conduct..."
271,2,1,1,18,"[SlNo, Tehsil, Block, Village, Latitude, Longitude, Potential of Hydrogen (pH), Electric Conduct..."
15,2,1,1,17,"[SlNo, Tehsil, Block, Latitude, Longitude, Potential of Hydrogen (pH), Electric Conductivity (μS..."
16,2,1,1,17,"[SlNo, Tehsil, Block, Latitude, Longitude, Potential of Hydrogen (pH), Electric Conductivity (μS..."
71,2,1,1,17,"[SlNo, Tehsil, Block, Latitude, Longitude, Data Acquisition Time, Potential of Hydrogen (pH), El..."
72,2,1,1,17,"[SlNo, Data Acquisition Time, Potential of Hydrogen (pH), Electric Conductivity (μS/cm), Total D..."
87,2,1,1,17,"[SlNo, Tehsil, Block, Latitude, Longitude, Data Acquisition Time, Potential of Hydrogen (pH), El..."
91,2,1,1,17,"[SlNo, Tehsil, Block, Latitude, Longitude, Potential of Hydrogen (pH), Electric Conductivity (μS..."



DETAILS OF FIVE MOST VARIABLE GROUPS

Station-date group: ('Sakhigopal', 'Odisha', 'PURI', datetime.date(2017, 5, 25))


,Station,State,District,_date_only,_source_file,SlNo,Agency,Latitude,Longitude,Data Acquisition Time,...,Carbonate (mg/L),Bicarbonate (mg/L),Total Alkalinity (mg/L as CaCO3),Chloride (mg/L),Sulphate (mg/L),Total Hardness (mgCaCO3/L),Calcium (mg/L),Magnesium (mg/L),Sodium (mg/L),Iron(mg/L)
125157,Sakhigopal,Odisha,PURI,2017-05-25,gwq_chemical_parameter_manual_cgwb_od_1961_2025,13264,CGWB,19.82361111,85.86500000,25-05-2017 00:00,...,24,232,230.16,411,97,307,34,55,287.7,0.372
125158,Sakhigopal,Odisha,PURI,2017-05-25,gwq_chemical_parameter_manual_cgwb_od_1961_2025,13265,CGWB,19.94138889,85.83222222,25-05-2017 00:00,...,,220,180.33,378,102,238,38,35,269.4,0.879



Station-date group: ('Ahmedpur', 'Punjab', 'ROPAR', datetime.date(2015, 5, 25))


,Station,State,District,_date_only,_source_file,SlNo,Agency,Latitude,Longitude,Data Acquisition Time,...,Carbonate (mg/L),Bicarbonate (mg/L),Total Alkalinity (mg/L as CaCO3),Chloride (mg/L),Sulphate (mg/L),Total Hardness (mgCaCO3/L),Calcium (mg/L),Magnesium (mg/L),Sodium (mg/L),Iron(mg/L)
127371,Ahmedpur,Punjab,ROPAR,2015-05-25,gwq_chemical_parameter_manual_cgwb_pb_1961_2025,78,CGWB,31.00000000,76.57500000,25-05-2015 00:00,...,,,,,,,,,,0.038
127372,Ahmedpur,Punjab,ROPAR,2015-05-25,gwq_chemical_parameter_manual_cgwb_pb_1961_2025,79,CGWB,31.21250000,76.55055556,25-05-2015 00:00,...,0,345,282.79,174,135,423,74,57.96,172,


... (2 outputs left out)


Station-date group: ('Paniparul', 'West Bengal', 'East Medinipur', datetime.date(2017, 5, 25))


,Station,State,District,_date_only,_source_file,SlNo,Agency,Latitude,Longitude,Data Acquisition Time,...,Bicarbonate (mg/L),Total Alkalinity (mg/L as CaCO3),Chloride (mg/L),Sulphate (mg/L),Silica(mg/L),Total Hardness (mgCaCO3/L),Calcium (mg/L),Magnesium (mg/L),Sodium (mg/L),Iron(mg/L)
172995,Paniparul,West Bengal,East Medinipur,2017-05-25,gwq_chemical_parameter_manual_cgwb_wb_1961_2025,5931,CGWB,21.85972222,87.51666667,25-05-2017 00:00,...,140,145,39,,23,185,30,27,33,7.6
172996,Paniparul,West Bengal,East Medinipur,2017-05-25,gwq_chemical_parameter_manual_cgwb_wb_1961_2025,5932,CGWB,21.75333333,87.53388889,25-05-2017 00:00,...,146,180,121,3.3,13,285,26,53,57,0.21



Station-date group: ('Alampur', 'Andhra Pradesh', 'ANANTAPUR', datetime.date(2017, 5, 25))


,Station,State,District,_date_only,_source_file,SlNo,Agency,Latitude,Longitude,Data Acquisition Time,...,Carbonate (mg/L),Bicarbonate (mg/L),Total Alkalinity (mg/L as CaCO3),Chloride (mg/L),Sulphate (mg/L),Silica(mg/L),Total Hardness (mgCaCO3/L),Calcium (mg/L),Magnesium (mg/L),Sodium (mg/L)
854,Alampur,Andhra Pradesh,ANANTAPUR,2017-05-25,gwq_chemical_parameter_manual_cgwb_ap_1961_2025,221,CGWB,14.19166667,78.16388889,25-05-2017 00:00,...,,702,575.41,553,200,,1000,216,112.1,260
855,Alampur,Andhra Pradesh,ANANTAPUR,2017-05-25,gwq_chemical_parameter_manual_cgwb_ap_1961_2025,222,CGWB,26.02944444,78.79722222,25-05-2017 00:00,...,0,472,,25,12,34,240,44,31.66,103


In [14]:
repeated_records = combined_df[
    combined_df.duplicated(
        subset=station_date_keys,
        keep=False
    )
].copy()

group_details = []

for key, group in repeated_records.groupby(
    station_date_keys,
    dropna=False,
    sort=False
):
    varying_columns = [
        col for col in reference_columns
        if group[col].nunique(dropna=False) > 1
    ]

    group_details.append({
        "_key": key,
        "Records": len(group),
        "Different_Agencies": group["Agency"].nunique(),
        "Different_Source_Files": group["_source_file"].nunique(),
        "Varying_Column_Count": len(varying_columns),
        "Varying_Columns": varying_columns
    })

repeated_group_audit = pd.DataFrame(group_details)

print("REPEATED GROUP ANALYSIS")
print("=" * 55)
print("Groups with differences in at least one original column:",
      (repeated_group_audit["Varying_Column_Count"] > 0).sum())

print("Groups with different agencies:",
      (repeated_group_audit["Different_Agencies"] > 1).sum())

display(
    repeated_group_audit.drop(columns="_key")
    .sort_values(
        ["Varying_Column_Count", "Records"],
        ascending=False
    )
    .head(25)
)

print("\nDETAILS OF FIVE MOST VARIABLE GROUPS")

for _, item in repeated_group_audit.sort_values(
    ["Varying_Column_Count", "Records"],
    ascending=False
).head(5).iterrows():

    key = item["_key"]

    mask = pd.Series(True, index=combined_df.index)

    for column, value in zip(station_date_keys, key):
        mask &= combined_df[column].eq(value)

    group = combined_df.loc[mask]

    detail_columns = list(dict.fromkeys(
        station_date_keys
        + ["_source_file", "SlNo", "Agency",
           "Latitude", "Longitude", "Data Acquisition Time"]
        + item["Varying_Columns"]
    ))

    print("\nStation-date group:", key)
    display(group[detail_columns])

REPEATED GROUP ANALYSIS
Groups with differences in at least one original column: 280
Groups with different agencies: 0


,Records,Different_Agencies,Different_Source_Files,Varying_Column_Count,Varying_Columns
194,2,1,1,18,"[SlNo, Tehsil, Block, Latitude, Longitude, Potential of Hydrogen (pH), Electric Conductivity (μS..."
196,2,1,1,18,"[SlNo, Tehsil, Block, Village, Latitude, Longitude, Potential of Hydrogen (pH), Electric Conduct..."
266,2,1,1,18,"[SlNo, Tehsil, Block, Village, Latitude, Longitude, Potential of Hydrogen (pH), Electric Conduct..."
271,2,1,1,18,"[SlNo, Tehsil, Block, Village, Latitude, Longitude, Potential of Hydrogen (pH), Electric Conduct..."
15,2,1,1,17,"[SlNo, Tehsil, Block, Latitude, Longitude, Potential of Hydrogen (pH), Electric Conductivity (μS..."
16,2,1,1,17,"[SlNo, Tehsil, Block, Latitude, Longitude, Potential of Hydrogen (pH), Electric Conductivity (μS..."
71,2,1,1,17,"[SlNo, Tehsil, Block, Latitude, Longitude, Data Acquisition Time, Potential of Hydrogen (pH), El..."
72,2,1,1,17,"[SlNo, Data Acquisition Time, Potential of Hydrogen (pH), Electric Conductivity (μS/cm), Total D..."
87,2,1,1,17,"[SlNo, Tehsil, Block, Latitude, Longitude, Data Acquisition Time, Potential of Hydrogen (pH), El..."
91,2,1,1,17,"[SlNo, Tehsil, Block, Latitude, Longitude, Potential of Hydrogen (pH), Electric Conductivity (μS..."



DETAILS OF FIVE MOST VARIABLE GROUPS

Station-date group: ('Sakhigopal', 'Odisha', 'PURI', datetime.date(2017, 5, 25))


,Station,State,District,_date_only,_source_file,SlNo,Agency,Latitude,Longitude,Data Acquisition Time,...,Carbonate (mg/L),Bicarbonate (mg/L),Total Alkalinity (mg/L as CaCO3),Chloride (mg/L),Sulphate (mg/L),Total Hardness (mgCaCO3/L),Calcium (mg/L),Magnesium (mg/L),Sodium (mg/L),Iron(mg/L)
125157,Sakhigopal,Odisha,PURI,2017-05-25,gwq_chemical_parameter_manual_cgwb_od_1961_2025,13264,CGWB,19.82361111,85.86500000,25-05-2017 00:00,...,24,232,230.16,411,97,307,34,55,287.7,0.372
125158,Sakhigopal,Odisha,PURI,2017-05-25,gwq_chemical_parameter_manual_cgwb_od_1961_2025,13265,CGWB,19.94138889,85.83222222,25-05-2017 00:00,...,,220,180.33,378,102,238,38,35,269.4,0.879



Station-date group: ('Ahmedpur', 'Punjab', 'ROPAR', datetime.date(2015, 5, 25))


,Station,State,District,_date_only,_source_file,SlNo,Agency,Latitude,Longitude,Data Acquisition Time,...,Carbonate (mg/L),Bicarbonate (mg/L),Total Alkalinity (mg/L as CaCO3),Chloride (mg/L),Sulphate (mg/L),Total Hardness (mgCaCO3/L),Calcium (mg/L),Magnesium (mg/L),Sodium (mg/L),Iron(mg/L)
127371,Ahmedpur,Punjab,ROPAR,2015-05-25,gwq_chemical_parameter_manual_cgwb_pb_1961_2025,78,CGWB,31.00000000,76.57500000,25-05-2015 00:00,...,,,,,,,,,,0.038
127372,Ahmedpur,Punjab,ROPAR,2015-05-25,gwq_chemical_parameter_manual_cgwb_pb_1961_2025,79,CGWB,31.21250000,76.55055556,25-05-2015 00:00,...,0,345,282.79,174,135,423,74,57.96,172,


... (2 outputs left out)


Station-date group: ('Paniparul', 'West Bengal', 'East Medinipur', datetime.date(2017, 5, 25))


,Station,State,District,_date_only,_source_file,SlNo,Agency,Latitude,Longitude,Data Acquisition Time,...,Bicarbonate (mg/L),Total Alkalinity (mg/L as CaCO3),Chloride (mg/L),Sulphate (mg/L),Silica(mg/L),Total Hardness (mgCaCO3/L),Calcium (mg/L),Magnesium (mg/L),Sodium (mg/L),Iron(mg/L)
172995,Paniparul,West Bengal,East Medinipur,2017-05-25,gwq_chemical_parameter_manual_cgwb_wb_1961_2025,5931,CGWB,21.85972222,87.51666667,25-05-2017 00:00,...,140,145,39,,23,185,30,27,33,7.6
172996,Paniparul,West Bengal,East Medinipur,2017-05-25,gwq_chemical_parameter_manual_cgwb_wb_1961_2025,5932,CGWB,21.75333333,87.53388889,25-05-2017 00:00,...,146,180,121,3.3,13,285,26,53,57,0.21



Station-date group: ('Alampur', 'Andhra Pradesh', 'ANANTAPUR', datetime.date(2017, 5, 25))


,Station,State,District,_date_only,_source_file,SlNo,Agency,Latitude,Longitude,Data Acquisition Time,...,Carbonate (mg/L),Bicarbonate (mg/L),Total Alkalinity (mg/L as CaCO3),Chloride (mg/L),Sulphate (mg/L),Silica(mg/L),Total Hardness (mgCaCO3/L),Calcium (mg/L),Magnesium (mg/L),Sodium (mg/L)
854,Alampur,Andhra Pradesh,ANANTAPUR,2017-05-25,gwq_chemical_parameter_manual_cgwb_ap_1961_2025,221,CGWB,14.19166667,78.16388889,25-05-2017 00:00,...,,702,575.41,553,200,,1000,216,112.1,260
855,Alampur,Andhra Pradesh,ANANTAPUR,2017-05-25,gwq_chemical_parameter_manual_cgwb_ap_1961_2025,222,CGWB,26.02944444,78.79722222,25-05-2017 00:00,...,0,472,,25,12,34,240,44,31.66,103


In [15]:
geo_df = combined_df.copy()

geo_df["_latitude_num"] = pd.to_numeric(
    geo_df["Latitude"], errors="coerce"
)

geo_df["_longitude_num"] = pd.to_numeric(
    geo_df["Longitude"], errors="coerce"
)

lat = geo_df["_latitude_num"]
lon = geo_df["_longitude_num"]

both_present = lat.notna() & lon.notna()

world_valid = (
    lat.between(-90, 90)
    & lon.between(-180, 180)
)

india_envelope = (
    lat.between(5, 39)
    & lon.between(67, 99)
)

print("GEOGRAPHIC COORDINATE AUDIT")
print("=" * 55)

print("Records with missing latitude:", lat.isna().sum())
print("Records with missing longitude:", lon.isna().sum())
print("Records with both coordinates:", both_present.sum())

print("Records outside valid world coordinate ranges:",
      (~world_valid).sum())

print("Records with zero latitude or longitude:",
      ((lat == 0) | (lon == 0)).sum())

print("Records outside broad India envelope:",
      (both_present & ~india_envelope).sum())

print("\nCOORDINATE RANGES")
print("Latitude minimum:", lat.min())
print("Latitude maximum:", lat.max())
print("Longitude minimum:", lon.min())
print("Longitude maximum:", lon.max())

print("\nEXAMPLES OUTSIDE BROAD INDIA ENVELOPE")

outside_india = geo_df[
    both_present & ~india_envelope
]

display(
    outside_india[
        ["Station", "State", "District",
         "Latitude", "Longitude"]
    ].drop_duplicates().head(30)
)

GEOGRAPHIC COORDINATE AUDIT
Records with missing latitude: 152
Records with missing longitude: 152
Records with both coordinates: 174483
Records outside valid world coordinate ranges: 152
Records with zero latitude or longitude: 0
Records outside broad India envelope: 0

COORDINATE RANGES
Latitude minimum: 8.08333333
Latitude maximum: 34.65027778
Longitude minimum: 68.53333333
Longitude maximum: 96.13333333

EXAMPLES OUTSIDE BROAD INDIA ENVELOPE


,Station,State,District,Latitude,Longitude


In [16]:
state_code_audit = (
    combined_df.groupby("State LGD Code", dropna=False)
    .agg(
        Distinct_State_Names=("State", "nunique"),
        State_Names=("State", lambda x: sorted(set(x))),
        Records=("State", "size")
    )
    .reset_index()
)

state_name_audit = (
    combined_df.groupby("State", dropna=False)
    .agg(
        Distinct_State_Codes=("State LGD Code", "nunique"),
        State_Codes=("State LGD Code", lambda x: sorted(set(x)))
    )
    .reset_index()
)

district_code_audit = (
    combined_df.groupby("District LGD Code", dropna=False)
    .agg(
        Distinct_Districts=("District", "nunique"),
        District_Names=("District", lambda x: sorted(set(x))),
        Distinct_States=("State", "nunique"),
        State_Names=("State", lambda x: sorted(set(x))),
        Records=("District", "size")
    )
    .reset_index()
)

district_name_audit = (
    combined_df.groupby(["State", "District"], dropna=False)
    .agg(
        Distinct_District_Codes=("District LGD Code", "nunique"),
        District_Codes=("District LGD Code",
                        lambda x: sorted(set(x)))
    )
    .reset_index()
)

print("STATE CODES MAPPED TO MULTIPLE STATE NAMES")
display(
    state_code_audit[
        state_code_audit["Distinct_State_Names"] > 1
    ]
)

print("\nSTATE NAMES MAPPED TO MULTIPLE CODES")
display(
    state_name_audit[
        state_name_audit["Distinct_State_Codes"] > 1
    ]
)

print("\nDISTRICT CODES MAPPED TO MULTIPLE DISTRICTS OR STATES")
display(
    district_code_audit[
        (district_code_audit["Distinct_Districts"] > 1)
        | (district_code_audit["Distinct_States"] > 1)
    ]
)

print("\nSTATE-DISTRICT PAIRS WITH MULTIPLE DISTRICT CODES")
display(
    district_name_audit[
        district_name_audit["Distinct_District_Codes"] > 1
    ]
)

STATE CODES MAPPED TO MULTIPLE STATE NAMES


,State LGD Code,Distinct_State_Names,State_Names,Records



STATE NAMES MAPPED TO MULTIPLE CODES


,State,Distinct_State_Codes,State_Codes



DISTRICT CODES MAPPED TO MULTIPLE DISTRICTS OR STATES


,District LGD Code,Distinct_Districts,District_Names,Distinct_States,State_Names,Records
5,104,2,"[JALORE, Jalore]",1,[Rajasthan],181
23,120,2,"[Allahabad, PRAYGRAJ]",1,[Uttar Pradesh],178
86,179,2,"[Bhadohi, SANT RAVIDAS NAGAR]",1,[Uttar Pradesh],49
193,310,2,"[DAKSHIN DINAJPUR, South Dinajpur]",1,[West Bengal],134
194,311,2,"[NORTH DINAJPUR, UTTAR DINAJPUR]",1,[West Bengal],200
210,327,2,"[EAST SINGHBHUM (Jamshedpur), PURBI SINGHBHUM]",1,[Jharkhand],94
225,343,2,"[PASHCHIMI SINGHBHUM, WEST SINGHBHUM (Chaibasa)]",1,[Jharkhand],150
333,441,2,"[Banaskantha, DAMAN]",1,[Gujarat],320
374,483,2,"[MUMBAI CITY, Mumbai Suburban]",1,[Maharashtra],185
403,509,2,"[KHAMMAM, Khammam]",2,"[Andhra Pradesh, Telangana]",730



STATE-DISTRICT PAIRS WITH MULTIPLE DISTRICT CODES


,State,District,Distinct_District_Codes,District_Codes


In [17]:
serial_audit = []

for name, df in datasets.items():

    serials = df["SlNo"].astype(str).str.strip()

    serial_audit.append({
        "Dataset": name,
        "Total_Records": len(df),
        "Unique_Serial_Numbers": serials.nunique(),
        "Duplicate_Serial_Rows": serials.duplicated(keep=False).sum(),
        "Blank_Serial_Numbers": serials.eq("").sum()
    })

serial_audit_df = pd.DataFrame(serial_audit)

print("SERIAL NUMBER AUDIT")
display(serial_audit_df)

print("\nAGENCY SUMMARY")

agency_summary = (
    combined_df.groupby(["State", "Agency"], dropna=False)
    .agg(
        Records=("Station", "size"),
        Unique_Stations=("Station", "nunique")
    )
    .reset_index()
    .sort_values("Records", ascending=False)
)

print("Total distinct agencies:", combined_df["Agency"].nunique())
print("States with multiple agencies:",
      (combined_df.groupby("State")["Agency"].nunique() > 1).sum())

display(agency_summary.head(30))

print("\nIDENTIFIER WHITESPACE AUDIT")

identifier_columns = [
    "SlNo", "Station", "Agency",
    "State LGD Code", "State",
    "District LGD Code", "District"
]

whitespace_audit = []

for column in identifier_columns:
    values = combined_df[column].astype(str)

    whitespace_audit.append({
        "Column": column,
        "Leading_or_Trailing_Whitespace":
            values.ne(values.str.strip()).sum(),
        "Blank_Values": values.str.strip().eq("").sum()
    })

display(pd.DataFrame(whitespace_audit))

SERIAL NUMBER AUDIT


,Dataset,Total_Records,Unique_Serial_Numbers,Duplicate_Serial_Rows,Blank_Serial_Numbers
0,gwq_chemical_parameter_manual_cgwb_an_1961_2025,634,634,0,0
1,gwq_chemical_parameter_manual_cgwb_ap_1961_2025,10950,10950,0,0
2,gwq_chemical_parameter_manual_cgwb_ar_1961_2025,104,104,0,0
3,gwq_chemical_parameter_manual_cgwb_as_1961_2025,1874,1874,0,0
4,gwq_chemical_parameter_manual_cgwb_br_1961_2025,3112,3112,0,0
5,gwq_chemical_parameter_manual_cgwb_cg_1961_2025,10350,10350,0,0
6,gwq_chemical_parameter_manual_cgwb_ch_1961_2025,36,36,0,0
7,gwq_chemical_parameter_manual_cgwb_dl_1961_2025,1169,1169,0,0
8,gwq_chemical_parameter_manual_cgwb_dn_1961_2025,169,169,0,0
9,gwq_chemical_parameter_manual_cgwb_ga_1961_2025,674,674,0,0



AGENCY SUMMARY
Total distinct agencies: 1
States with multiple agencies: 0


,State,Agency,Records,Unique_Stations
17,Madhya Pradesh,CGWB,19230,1670
18,Maharashtra,CGWB,18249,1758
21,Odisha,CGWB,15400,1897
15,Karnataka,CGWB,13007,2007
24,Rajasthan,CGWB,12350,1606
1,Andhra Pradesh,CGWB,10950,1538
6,Chhattisgarh,CGWB,10350,1255
10,Gujarat,CGWB,9807,958
16,Kerala,CGWB,9473,1048
25,Tamil Nadu,CGWB,8419,1215



IDENTIFIER WHITESPACE AUDIT


,Column,Leading_or_Trailing_Whitespace,Blank_Values
0,SlNo,0,0
1,Station,33,0
2,Agency,0,0
3,State LGD Code,0,0
4,State,0,0
5,District LGD Code,0,0
6,District,0,0


In [18]:
lat = pd.to_numeric(combined_df["Latitude"], errors="coerce")
lon = pd.to_numeric(combined_df["Longitude"], errors="coerce")

both_present = lat.notna() & lon.notna()
both_missing = lat.isna() & lon.isna()
one_missing = lat.isna() ^ lon.isna()

world_valid = (
    lat.between(-90, 90)
    & lon.between(-180, 180)
)

india_envelope = (
    lat.between(5, 39)
    & lon.between(67, 99)
)

print("CORRECTED COORDINATE AUDIT")
print("=" * 50)

print("Both coordinates present:", both_present.sum())
print("Both coordinates missing:", both_missing.sum())
print("Only one coordinate missing:", one_missing.sum())

print("Invalid populated world coordinates:",
      (both_present & ~world_valid).sum())

print("Populated coordinates outside broad India envelope:",
      (both_present & ~india_envelope).sum())

print("\nCOORDINATE COMPLETENESS BY STATE")

coordinate_summary = (
    combined_df.assign(
        _lat_num=lat,
        _lon_num=lon,
        _coordinates_present=both_present
    )
    .groupby("State")
    .agg(
        Records=("Station", "size"),
        Coordinates_Present=("_coordinates_present", "sum")
    )
    .reset_index()
)

coordinate_summary["Coordinate_Coverage_%"] = (
    coordinate_summary["Coordinates_Present"]
    / coordinate_summary["Records"] * 100
).round(2)

display(
    coordinate_summary.sort_values(
        "Coordinate_Coverage_%"
    )
)

CORRECTED COORDINATE AUDIT
Both coordinates present: 174483
Both coordinates missing: 152
Only one coordinate missing: 0
Invalid populated world coordinates: 0
Populated coordinates outside broad India envelope: 0

COORDINATE COMPLETENESS BY STATE


,State,Records,Coordinates_Present,Coordinate_Coverage_%
1,Andhra Pradesh,10950,10832,98.92
25,Tamil Nadu,8419,8397,99.74
26,Telangana,5072,5062,99.80
21,Odisha,15400,15399,99.99
24,Rajasthan,12350,12349,99.99
0,Andaman And Nicobar Islands,634,634,100.00
2,Arunachal Pradesh,104,104,100.00
3,Assam,1874,1874,100.00
8,Delhi,1169,1169,100.00
9,Goa,674,674,100.00


In [19]:
import numpy as np

coordinate_df = combined_df.copy()

coordinate_df["_lat_num"] = pd.to_numeric(
    coordinate_df["Latitude"], errors="coerce"
)

coordinate_df["_lon_num"] = pd.to_numeric(
    coordinate_df["Longitude"], errors="coerce"
)

coordinate_candidates = station_summary[
    (station_summary["Unique_Latitudes"] > 1)
    | (station_summary["Unique_Longitudes"] > 1)
]

candidate_keys = set(
    tuple(row)
    for row in coordinate_candidates[station_keys].itertuples(
        index=False, name=None
    )
)

def maximum_coordinate_distance_km(group):

    coords = (
        group[["_lat_num", "_lon_num"]]
        .dropna()
        .drop_duplicates()
        .to_numpy(dtype=float)
    )

    if len(coords) < 2:
        return 0.0

    lat_rad = np.radians(coords[:, 0])
    lon_rad = np.radians(coords[:, 1])

    dlat = lat_rad[:, None] - lat_rad[None, :]
    dlon = lon_rad[:, None] - lon_rad[None, :]

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat_rad[:, None])
        * np.cos(lat_rad[None, :])
        * np.sin(dlon / 2) ** 2
    )

    distances = (
        6371 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    )

    return round(float(distances.max()), 3)


coordinate_consistency = []

for key, group in coordinate_df.groupby(
    station_keys, dropna=False, sort=False
):

    if key not in candidate_keys:
        continue

    coordinate_consistency.append({
        "Station": key[0],
        "State": key[1],
        "District": key[2],
        "Records": len(group),
        "Unique_Coordinate_Pairs": len(
            group[["_lat_num", "_lon_num"]]
            .dropna()
            .drop_duplicates()
        ),
        "Maximum_Coordinate_Distance_km":
            maximum_coordinate_distance_km(group)
    })

coordinate_consistency_df = pd.DataFrame(
    coordinate_consistency
)

print("STATION COORDINATE CONSISTENCY")
print("=" * 50)

print("Provisional station identities with multiple coordinates:",
      len(coordinate_consistency_df))

display(
    coordinate_consistency_df.sort_values(
        "Maximum_Coordinate_Distance_km",
        ascending=False
    ).head(30)
)

STATION COORDINATE CONSISTENCY
Provisional station identities with multiple coordinates: 350


,Station,State,District,Records,Unique_Coordinate_Pairs,Maximum_Coordinate_Distance_km
2,Alampur,Andhra Pradesh,ANANTAPUR,39,3,1318.083
12,Madhudi,Andhra Pradesh,ANANTAPUR,8,2,122.635
260,Kalamadugu,Telangana,ADILABAD,15,2,94.117
342,Suisa,West Bengal,PURULIYA,15,2,88.223
229,Hazipur,Punjab,HOSHIARPUR,22,2,87.233
99,Mahendrapur,Bihar,KATIHAR,4,2,75.448
164,Devri,Chhattisgarh,RAIPUR,15,2,70.807
110,Mokama,Bihar,PATNA,8,2,70.549
301,Helencha P.s.,West Bengal,NORTH 24 PARGANAS,5,2,67.405
252,Bantwaram,Telangana,RANGA REDDY,8,2,67.285


In [20]:
station_format_df = combined_df[
    ["Station", "State", "District"]
].copy()

station_format_df["_normalized_station"] = (
    station_format_df["Station"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.casefold()
)

whitespace_rows = combined_df[
    combined_df["Station"].astype(str).ne(
        combined_df["Station"].astype(str).str.strip()
    )
]

print("STATION FORMATTING AUDIT")
print("=" * 50)

print("Rows with leading/trailing whitespace:",
      len(whitespace_rows))

display(
    whitespace_rows[
        ["Station", "State", "District"]
    ].drop_duplicates()
)

# Different original station labels with the same normalized label
station_alias_audit = (
    station_format_df
    .groupby("_normalized_station")
    .agg(
        Original_Name_Count=("Station", "nunique"),
        Original_Names=("Station",
                        lambda x: sorted(set(x))),
        States=("State", "nunique"),
        Districts=("District", "nunique")
    )
    .reset_index()
)

station_alias_audit = station_alias_audit[
    station_alias_audit["Original_Name_Count"] > 1
]

print("\nNORMALIZED STATION-NAME COLLISIONS:",
      len(station_alias_audit))

display(station_alias_audit.head(40))

STATION FORMATTING AUDIT
Rows with leading/trailing whitespace: 33


,Station,State,District
18816,Chhal,Chhattisgarh,RAIGARH
159890,Aliya Q,Uttar Pradesh,SITAPUR
160446,Banda Q,Uttar Pradesh,Shahjhanpur
164499,Mehdawal Q,Uttar Pradesh,SANT KABIR NAGAR
166764,Tamkuhi Q,Uttar Pradesh,KUSHINAGAR



NORMALIZED STATION-NAME COLLISIONS: 142


,_normalized_station,Original_Name_Count,Original_Names,States,Districts
497,akola,2,"[AKOLA, Akola]",3,3
537,alamsar,2,"[ALAMSAR, Alamsar]",1,1
580,aligarh,2,"[ALIGARH, Aligarh]",2,2
899,anantpura,2,"[ANANTPURA, Anantpura]",1,2
1241,aspalsar,2,"[ASPALSAR, Aspalsar]",1,1
1539,badagaon,2,"[BADAGAON, Badagaon]",2,2
2093,bamla,2,"[BAMLA, Bamla]",2,2
2302,baori,2,"[BAORI, Baori]",1,1
2307,bapi,2,"[BAPI, Bapi]",1,1
2456,bari pada,2,"[Bari Pada, Bari pada]",1,1


In [21]:
state_code_audit = (
    combined_df.groupby("State LGD Code", dropna=False)
    .agg(
        State_Count=("State", "nunique"),
        State_Names=("State", lambda x: sorted(set(x))),
        Records=("State", "size")
    )
    .reset_index()
)

district_code_audit = (
    combined_df.groupby("District LGD Code", dropna=False)
    .agg(
        District_Count=("District", "nunique"),
        District_Names=("District", lambda x: sorted(set(x))),
        State_Count=("State", "nunique"),
        State_Names=("State", lambda x: sorted(set(x))),
        Records=("District", "size")
    )
    .reset_index()
)

district_name_audit = (
    combined_df.groupby(["State", "District"], dropna=False)
    .agg(
        Code_Count=("District LGD Code", "nunique"),
        Codes=("District LGD Code",
               lambda x: sorted(set(x)))
    )
    .reset_index()
)

print("STATE CODE CONFLICTS")
display(state_code_audit[state_code_audit["State_Count"] > 1])

print("\nDISTRICT CODE CONFLICTS")
display(
    district_code_audit[
        (district_code_audit["District_Count"] > 1)
        | (district_code_audit["State_Count"] > 1)
    ]
)

print("\nDISTRICT NAMES ASSOCIATED WITH MULTIPLE CODES")
display(district_name_audit[district_name_audit["Code_Count"] > 1])

print("\nRECORDS WITH DISTRICT CODE 9999")

display(
    combined_df[
        combined_df["District LGD Code"].astype(str).str.strip() == "9999"
    ][
        ["State", "District", "District LGD Code",
         "State LGD Code", "Station"]
    ].drop_duplicates()
)

STATE CODE CONFLICTS


,State LGD Code,State_Count,State_Names,Records



DISTRICT CODE CONFLICTS


,District LGD Code,District_Count,District_Names,State_Count,State_Names,Records
5,104,2,"[JALORE, Jalore]",1,[Rajasthan],181
23,120,2,"[Allahabad, PRAYGRAJ]",1,[Uttar Pradesh],178
86,179,2,"[Bhadohi, SANT RAVIDAS NAGAR]",1,[Uttar Pradesh],49
193,310,2,"[DAKSHIN DINAJPUR, South Dinajpur]",1,[West Bengal],134
194,311,2,"[NORTH DINAJPUR, UTTAR DINAJPUR]",1,[West Bengal],200
210,327,2,"[EAST SINGHBHUM (Jamshedpur), PURBI SINGHBHUM]",1,[Jharkhand],94
225,343,2,"[PASHCHIMI SINGHBHUM, WEST SINGHBHUM (Chaibasa)]",1,[Jharkhand],150
333,441,2,"[Banaskantha, DAMAN]",1,[Gujarat],320
374,483,2,"[MUMBAI CITY, Mumbai Suburban]",1,[Maharashtra],185
403,509,2,"[KHAMMAM, Khammam]",2,"[Andhra Pradesh, Telangana]",730



DISTRICT NAMES ASSOCIATED WITH MULTIPLE CODES


,State,District,Code_Count,Codes



RECORDS WITH DISTRICT CODE 9999


,State,District,District LGD Code,State LGD Code,Station
132469,Puducherry,Yanam,9999,34,Bandurlanka-DW13
167433,West Bengal,Calcutta,9999,19,Bagbazar
167669,West Bengal,Calcutta,9999,19,Ballyganj Maidan
167794,West Bengal,KOLKATA,9999,19,Bantala-Nutanhat
168112,West Bengal,KOLKATA,9999,19,Beleghata
169432,West Bengal,Calcutta,9999,19,Dumdum
169678,West Bengal,Calcutta,9999,19,Garden Reach
170663,West Bengal,Calcutta,9999,19,Jhautala
171157,West Bengal,KOLKATA,9999,19,Keorapukur
171520,West Bengal,Calcutta,9999,19,Kudghat


In [22]:
coordinate_flags = (
    coordinate_consistency_df[
        coordinate_consistency_df[
            "Maximum_Coordinate_Distance_km"
        ] > 25
    ]
    .sort_values(
        "Maximum_Coordinate_Distance_km",
        ascending=False
    )
)

print("STATION IDENTITIES WITH COORDINATE DIFFERENCES ABOVE 25 KM")
print("=" * 65)

print("Affected station identities:", len(coordinate_flags))

display(coordinate_flags)

print("\nDETAILED RECORDS FOR THE TOP 10 IDENTITIES")

for _, item in coordinate_flags.head(10).iterrows():

    mask = (
        combined_df["Station"].eq(item["Station"])
        & combined_df["State"].eq(item["State"])
        & combined_df["District"].eq(item["District"])
    )

    print(
        f"\n{item['Station']} | {item['State']} | "
        f"{item['District']} | "
        f"Maximum distance: "
        f"{item['Maximum_Coordinate_Distance_km']} km"
    )

    display(
        combined_df.loc[
            mask,
            [
                "Station",
                "State",
                "District",
                "Latitude",
                "Longitude",
                "Data Acquisition Time",
                "_source_file"
            ]
        ].sort_values("Data Acquisition Time")
    )

STATION IDENTITIES WITH COORDINATE DIFFERENCES ABOVE 25 KM
Affected station identities: 42


,Station,State,District,Records,Unique_Coordinate_Pairs,Maximum_Coordinate_Distance_km
2,Alampur,Andhra Pradesh,ANANTAPUR,39,3,1318.083
12,Madhudi,Andhra Pradesh,ANANTAPUR,8,2,122.635
260,Kalamadugu,Telangana,ADILABAD,15,2,94.117
342,Suisa,West Bengal,PURULIYA,15,2,88.223
229,Hazipur,Punjab,HOSHIARPUR,22,2,87.233
99,Mahendrapur,Bihar,KATIHAR,4,2,75.448
164,Devri,Chhattisgarh,RAIPUR,15,2,70.807
110,Mokama,Bihar,PATNA,8,2,70.549
301,Helencha P.s.,West Bengal,NORTH 24 PARGANAS,5,2,67.405
252,Bantwaram,Telangana,RANGA REDDY,8,2,67.285



DETAILED RECORDS FOR THE TOP 10 IDENTITIES

Alampur | Andhra Pradesh | ANANTAPUR | Maximum distance: 1318.083 km


,Station,State,District,Latitude,Longitude,Data Acquisition Time,_source_file
859,Alampur,Andhra Pradesh,ANANTAPUR,14.18972222,78.18055556,20-05-2022 18:29,gwq_chemical_parameter_manual_cgwb_ap_1961_2025
822,Alampur,Andhra Pradesh,ANANTAPUR,26.02944444,78.79722222,25-05-2000 00:00,gwq_chemical_parameter_manual_cgwb_ap_1961_2025
823,Alampur,Andhra Pradesh,ANANTAPUR,14.19166667,78.16388889,25-05-2000 00:00,gwq_chemical_parameter_manual_cgwb_ap_1961_2025
824,Alampur,Andhra Pradesh,ANANTAPUR,26.02944444,78.79722222,25-05-2001 00:00,gwq_chemical_parameter_manual_cgwb_ap_1961_2025
825,Alampur,Andhra Pradesh,ANANTAPUR,14.19166667,78.16388889,25-05-2001 00:00,gwq_chemical_parameter_manual_cgwb_ap_1961_2025
826,Alampur,Andhra Pradesh,ANANTAPUR,26.02944444,78.79722222,25-05-2002 00:00,gwq_chemical_parameter_manual_cgwb_ap_1961_2025
827,Alampur,Andhra Pradesh,ANANTAPUR,14.19166667,78.16388889,25-05-2002 00:00,gwq_chemical_parameter_manual_cgwb_ap_1961_2025
828,Alampur,Andhra Pradesh,ANANTAPUR,14.19166667,78.16388889,25-05-2003 00:00,gwq_chemical_parameter_manual_cgwb_ap_1961_2025
829,Alampur,Andhra Pradesh,ANANTAPUR,14.19166667,78.16388889,25-05-2004 00:00,gwq_chemical_parameter_manual_cgwb_ap_1961_2025
830,Alampur,Andhra Pradesh,ANANTAPUR,26.02944444,78.79722222,25-05-2004 00:00,gwq_chemical_parameter_manual_cgwb_ap_1961_2025



Madhudi | Andhra Pradesh | ANANTAPUR | Maximum distance: 122.635 km


,Station,State,District,Latitude,Longitude,Data Acquisition Time,_source_file
6079,Madhudi,Andhra Pradesh,ANANTAPUR,13.81694444,77.02361111,25-05-2000 00:00,gwq_chemical_parameter_manual_cgwb_ap_1961_2025
6080,Madhudi,Andhra Pradesh,ANANTAPUR,14.25805556,78.06555556,25-05-2000 00:00,gwq_chemical_parameter_manual_cgwb_ap_1961_2025
6081,Madhudi,Andhra Pradesh,ANANTAPUR,14.25805556,78.06555556,25-05-2001 00:00,gwq_chemical_parameter_manual_cgwb_ap_1961_2025
6082,Madhudi,Andhra Pradesh,ANANTAPUR,13.81694444,77.02361111,25-05-2001 00:00,gwq_chemical_parameter_manual_cgwb_ap_1961_2025
6083,Madhudi,Andhra Pradesh,ANANTAPUR,13.81694444,77.02361111,25-05-2002 00:00,gwq_chemical_parameter_manual_cgwb_ap_1961_2025
6084,Madhudi,Andhra Pradesh,ANANTAPUR,14.25805556,78.06555556,25-05-2002 00:00,gwq_chemical_parameter_manual_cgwb_ap_1961_2025
6085,Madhudi,Andhra Pradesh,ANANTAPUR,13.81694444,77.02361111,25-05-2003 00:00,gwq_chemical_parameter_manual_cgwb_ap_1961_2025
6086,Madhudi,Andhra Pradesh,ANANTAPUR,14.25805556,78.06555556,25-05-2003 00:00,gwq_chemical_parameter_manual_cgwb_ap_1961_2025


... (12 outputs left out)


Helencha P.s. | West Bengal | NORTH 24 PARGANAS | Maximum distance: 67.405 km


,Station,State,District,Latitude,Longitude,Data Acquisition Time,_source_file
170202,Helencha P.s.,West Bengal,NORTH 24 PARGANAS,23.19000000,88.86000000,25-05-2002 00:00,gwq_chemical_parameter_manual_cgwb_wb_1961_2025
170203,Helencha P.s.,West Bengal,NORTH 24 PARGANAS,23.19000000,88.86000000,25-05-2003 00:00,gwq_chemical_parameter_manual_cgwb_wb_1961_2025
170204,Helencha P.s.,West Bengal,NORTH 24 PARGANAS,23.19000000,88.86000000,25-05-2016 00:00,gwq_chemical_parameter_manual_cgwb_wb_1961_2025
170205,Helencha P.s.,West Bengal,NORTH 24 PARGANAS,22.58638889,88.92055556,25-05-2016 00:00,gwq_chemical_parameter_manual_cgwb_wb_1961_2025
170206,Helencha P.s.,West Bengal,NORTH 24 PARGANAS,22.58638889,88.92055556,25-05-2017 00:00,gwq_chemical_parameter_manual_cgwb_wb_1961_2025



Bantwaram | Telangana | RANGA REDDY | Maximum distance: 67.285 km


,Station,State,District,Latitude,Longitude,Data Acquisition Time,_source_file
154190,Bantwaram,Telangana,RANGA REDDY,17.44694444,77.70777778,25-05-2002 00:00,gwq_chemical_parameter_manual_cgwb_ts_1961_2025
154191,Bantwaram,Telangana,RANGA REDDY,17.49333333,78.34027778,25-05-2013 00:00,gwq_chemical_parameter_manual_cgwb_ts_1961_2025
154192,Bantwaram,Telangana,RANGA REDDY,17.49333333,78.34027778,25-05-2014 00:00,gwq_chemical_parameter_manual_cgwb_ts_1961_2025
154193,Bantwaram,Telangana,RANGA REDDY,17.49333333,78.34027778,25-05-2016 00:00,gwq_chemical_parameter_manual_cgwb_ts_1961_2025
154194,Bantwaram,Telangana,RANGA REDDY,17.49333333,78.34027778,25-05-2017 00:00,gwq_chemical_parameter_manual_cgwb_ts_1961_2025
154195,Bantwaram,Telangana,RANGA REDDY,17.49333333,78.34027778,25-05-2018 00:00,gwq_chemical_parameter_manual_cgwb_ts_1961_2025
154196,Bantwaram,Telangana,RANGA REDDY,17.49333333,78.34027778,30-05-2019 08:00,gwq_chemical_parameter_manual_cgwb_ts_1961_2025
154197,Bantwaram,Telangana,RANGA REDDY,17.49333333,78.34027778,30-05-2020 08:00,gwq_chemical_parameter_manual_cgwb_ts_1961_2025


In [23]:
import numpy as np

chemical_columns = reference_columns[19:]

chemical_quality_results = []

for column in chemical_columns:

    values = pd.to_numeric(
        combined_df[column].astype(str).str.strip(),
        errors="coerce"
    )

    valid = values.notna() & np.isfinite(values)
    valid_values = values[valid]

    negative_count = (valid_values < 0).sum()
    zero_count = (valid_values == 0).sum()

    if column == "Potential of Hydrogen (pH)":
        outside_ph_range = (
            (valid_values < 0) | (valid_values > 14)
        ).sum()
    else:
        outside_ph_range = None

    chemical_quality_results.append({
        "Parameter": column,
        "Available_Numeric_Values": len(valid_values),
        "Negative_Count": int(negative_count),
        "Zero_Count": int(zero_count),
        "Outside_pH_0_14": outside_ph_range,
        "Minimum": valid_values.min() if len(valid_values) else np.nan,
        "P01": valid_values.quantile(0.01) if len(valid_values) else np.nan,
        "Median": valid_values.median() if len(valid_values) else np.nan,
        "P99": valid_values.quantile(0.99) if len(valid_values) else np.nan,
        "Maximum": valid_values.max() if len(valid_values) else np.nan
    })

chemical_quality_df = pd.DataFrame(
    chemical_quality_results
)

print("CHEMICAL MEASUREMENT PLAUSIBILITY AUDIT")
print("=" * 60)

display(chemical_quality_df)

CHEMICAL MEASUREMENT PLAUSIBILITY AUDIT


,Parameter,Available_Numeric_Values,Negative_Count,Zero_Count,Outside_pH_0_14,Minimum,P01,Median,P99,Maximum
0,Potential of Hydrogen (pH),162729,0,4,6.0,0.000000,6.6800,7.870,8.97000,2.667000e+03
1,Electric Conductivity (μS/cm),173029,0,2,NaN,0.000000,65.0000,801.000,7727.20000,5.700676e+07
2,Total Dissolved Solids (mg/L),30151,0,6646,NaN,0.000000,0.0000,421.000,5586.00000,2.353897e+06
3,Carbonate (mg/L),120229,0,104729,NaN,0.000000,0.0000,0.000,96.00000,5.979960e+02
4,Bicarbonate (mg/L),147171,1,739,NaN,-6.000000,9.8000,256.000,915.00000,8.178300e+05
5,Total Alkalinity (mg/L as CaCO3),79936,0,3386,NaN,0.000000,0.0000,210.000,809.80000,7.909840e+03
6,Chloride (mg/L),162034,1,10,NaN,-645.000000,6.9132,72.000,1953.00000,3.094720e+05
7,Nitrate N (mgN/L),0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
8,Sulphate (mg/L),126328,2,5806,NaN,-0.466207,0.0000,35.000,990.00000,4.185500e+04
9,Phosphate(mg/L),8759,0,6134,NaN,0.000000,0.0000,0.000,1.05840,3.100000e+01


In [24]:
# Preserve each record's original row position within its CSV
combined_df["_source_row"] = (
    combined_df.groupby("_source_file").cumcount()
)

ph_column = "Potential of Hydrogen (pH)"

ph_values = pd.to_numeric(
    combined_df[ph_column].astype(str).str.strip(),
    errors="coerce"
)

ph_suspicious_mask = (
    ph_values.notna()
    & ((ph_values < 0) | (ph_values > 14) | (ph_values == 0))
)

ph_suspicious = combined_df.loc[
    ph_suspicious_mask,
    [
        "_source_file", "_source_row",
        "Station", "State", "District",
        "Data Acquisition Time",
        "Latitude", "Longitude",
        ph_column
    ]
].copy()

ph_suspicious[ph_column] = ph_values[ph_suspicious_mask]

print("SUSPICIOUS pH RECORDS")
print("=" * 55)
print("Total:", len(ph_suspicious))
print("Zero pH values:", (ph_values == 0).sum())
print("Values below 0:", (ph_values < 0).sum())
print("Values above 14:", (ph_values > 14).sum())

display(
    ph_suspicious.sort_values(ph_column)
)

SUSPICIOUS pH RECORDS
Total: 10
Zero pH values: 4
Values below 0: 0
Values above 14: 6


,_source_file,_source_row,Station,State,District,Data Acquisition Time,Latitude,Longitude,Potential of Hydrogen (pH)
18441,gwq_chemical_parameter_manual_cgwb_cg_1961_2025,1767,Biroda,Chhattisgarh,Kabirdham,30-05-2021 14:18,21.76722222,81.13055556,0.00
68066,gwq_chemical_parameter_manual_cgwb_kl_1961_2025,3403,Kokkothamangalam,Kerala,Alappuzha,25-05-2018 00:00,9.67972222,76.37416667,0.00
70306,gwq_chemical_parameter_manual_cgwb_kl_1961_2025,5643,Nedumkandam,Kerala,IDUKKI,25-05-2018 00:00,9.83777778,77.16055556,0.00
167101,gwq_chemical_parameter_manual_cgwb_wb_1961_2025,36,Aharrah,West Bengal,PURULIYA,25-05-2018 00:00,23.28750000,86.21250000,0.00
91375,gwq_chemical_parameter_manual_cgwb_mh_1961_2025,17239,Vile Parle West,Maharashtra,Mumbai Suburban,25-05-2018 00:00,19.09750000,72.84305556,76.00
142660,gwq_chemical_parameter_manual_cgwb_rj_1961_2025,10108,RAMGARH1,Rajasthan,ALWAR,25-05-2016 00:00,27.58333333,76.80972222,88.12
129635,gwq_chemical_parameter_manual_cgwb_pb_1961_2025,2341,Guru Sar-Pz,Punjab,BATHINDA,03-05-2023 10:52,29.99305556,75.09166667,738.00
25240,gwq_chemical_parameter_manual_cgwb_cg_1961_2025,8566,Sahaspur lohara,Chhattisgarh,Kabirdham,30-05-2021 14:42,21.83333333,81.12916667,796.00
39179,gwq_chemical_parameter_manual_cgwb_hp_1961_2025,300,Dehra Gopipur,Himachal Pradesh,KANGRA,25-05-2018 00:00,31.88333333,76.21666667,842.00
87489,gwq_chemical_parameter_manual_cgwb_mh_1961_2025,13353,Ranzani,Maharashtra,Solapur,20-05-2023 12:39,17.61666667,75.40000000,2667.00


In [25]:
extreme_records = []

for column in chemical_columns:

    values = pd.to_numeric(
        combined_df[column].astype(str).str.strip(),
        errors="coerce"
    )

    valid = values.notna() & np.isfinite(values)

    if not valid.any():
        continue

    for extreme_type, index in [
        ("Minimum", values[valid].idxmin()),
        ("Maximum", values[valid].idxmax())
    ]:

        row = combined_df.loc[index]

        extreme_records.append({
            "Parameter": column,
            "Extreme_Type": extreme_type,
            "Value": values.loc[index],
            "Station": row["Station"],
            "State": row["State"],
            "District": row["District"],
            "Date": row["Data Acquisition Time"],
            "Latitude": row["Latitude"],
            "Longitude": row["Longitude"],
            "Source_File": row["_source_file"],
            "Source_Row": row["_source_row"]
        })

extreme_records_df = pd.DataFrame(extreme_records)

print("CHEMICAL EXTREME RECORDS")
display(
    extreme_records_df.sort_values(
        ["Parameter", "Extreme_Type"]
    ).reset_index(drop=True)
)

print("\nNEGATIVE CHEMICAL MEASUREMENTS")

negative_records = []

for column in chemical_columns:

    values = pd.to_numeric(
        combined_df[column].astype(str).str.strip(),
        errors="coerce"
    )

    for index in values[values < 0].index:

        row = combined_df.loc[index]

        negative_records.append({
            "Parameter": column,
            "Value": values.loc[index],
            "Station": row["Station"],
            "State": row["State"],
            "District": row["District"],
            "Date": row["Data Acquisition Time"],
            "Source_File": row["_source_file"],
            "Source_Row": row["_source_row"]
        })

negative_records_df = pd.DataFrame(negative_records)

print("Total negative chemical measurements:",
      len(negative_records_df))

display(
    negative_records_df.sort_values(
        "Parameter"
    ).head(40)
)

CHEMICAL EXTREME RECORDS


,Parameter,Extreme_Type,Value,Station,State,District,Date,Latitude,Longitude,Source_File,Source_Row
0,Arsenic (mg/L),Maximum,3.454550e+02,Manu,Tripura,DHALAI,25-05-2017 00:00,24.20416667,91.84194444,gwq_chemical_parameter_manual_cgwb_tr_1961_2025,288
1,Arsenic (mg/L),Minimum,0.000000e+00,Annikat,Andaman And Nicobar Islands,South Andaman,25-05-2018 00:00,11.70861111,92.65000000,gwq_chemical_parameter_manual_cgwb_an_1961_2025,8
2,Bicarbonate (mg/L),Maximum,8.178300e+05,RAMGARH1,Rajasthan,ALWAR,25-05-2016 00:00,27.58333333,76.80972222,gwq_chemical_parameter_manual_cgwb_rj_1961_2025,10108
3,Bicarbonate (mg/L),Minimum,-6.000000e+00,Panjipara,West Bengal,NORTH DINAJPUR,25-05-2017 00:00,26.27611111,88.25694444,gwq_chemical_parameter_manual_cgwb_wb_1961_2025,5936
4,Cadmium (mg/L),Maximum,2.000000e-03,Phul,Punjab,BATHINDA,03-05-2023 11:46,30.32500000,75.24166667,gwq_chemical_parameter_manual_cgwb_pb_1961_2025,4288
5,Cadmium (mg/L),Minimum,0.000000e+00,Abdullapur,Haryana,PANCHKULA,12-06-2022 12:01,30.79472222,76.91638889,gwq_chemical_parameter_manual_cgwb_hr_1961_2025,1
6,Calcium (mg/L),Maximum,1.001660e+05,Dharapuram2,Tamil Nadu,Erode,25-05-2001 00:00,10.73888889,77.52916667,gwq_chemical_parameter_manual_cgwb_tn_1961_2025,1360
7,Calcium (mg/L),Minimum,0.000000e+00,Oating,Assam,GOLAGHAT,25-05-2018 00:00,26.43333333,94.00833333,gwq_chemical_parameter_manual_cgwb_as_1961_2025,1350
8,Carbonate (mg/L),Maximum,5.979960e+02,Thondi,Tamil Nadu,Ramanathapuram,25-05-2018 00:00,9.76666667,79.01666667,gwq_chemical_parameter_manual_cgwb_tn_1961_2025,6853
9,Carbonate (mg/L),Minimum,0.000000e+00,Annikat,Andaman And Nicobar Islands,South Andaman,25-05-2018 00:00,11.70861111,92.65000000,gwq_chemical_parameter_manual_cgwb_an_1961_2025,8



NEGATIVE CHEMICAL MEASUREMENTS
Total negative chemical measurements: 138


,Parameter,Value,Station,State,District,Date,Source_File,Source_Row
0,Bicarbonate (mg/L),-6.000000e+00,Panjipara,West Bengal,NORTH DINAJPUR,25-05-2017 00:00,gwq_chemical_parameter_manual_cgwb_wb_1961_2025,5936
1,Chloride (mg/L),-6.450000e+02,Panchberia,West Bengal,West Medinipur,25-05-2017 00:00,gwq_chemical_parameter_manual_cgwb_wb_1961_2025,5869
135,Iron(mg/L),-2.200000e-02,Mumra-Pz,Punjab,FARIDKOT,03-05-2023 12:51,gwq_chemical_parameter_manual_cgwb_pb_1961_2025,3866
134,Iron(mg/L),-2.000000e-02,Mohre Wala-Pz,Punjab,FEROZEPUR,04-05-2023 10:13,gwq_chemical_parameter_manual_cgwb_pb_1961_2025,3783
133,Iron(mg/L),-3.000000e-03,Matta,Punjab,FARIDKOT,03-05-2023 15:24,gwq_chemical_parameter_manual_cgwb_pb_1961_2025,3669
132,Iron(mg/L),-1.500000e-02,Dhilwan Kalan,Punjab,FARIDKOT,03-05-2023 15:17,gwq_chemical_parameter_manual_cgwb_pb_1961_2025,1787
131,Iron(mg/L),-1.700000e-02,Dal Singh Wala,Punjab,FARIDKOT,03-05-2023 15:36,gwq_chemical_parameter_manual_cgwb_pb_1961_2025,1476
130,Iron(mg/L),-1.900000e-02,Baja Khana,Punjab,FARIDKOT,03-05-2023 15:20,gwq_chemical_parameter_manual_cgwb_pb_1961_2025,357
136,Iron(mg/L),-7.000000e-03,Nangal-Pz,Punjab,FARIDKOT,03-05-2023 15:09,gwq_chemical_parameter_manual_cgwb_pb_1961_2025,3956
137,Iron(mg/L),-1.000000e-02,Sukhwala-Pz,Punjab,FARIDKOT,03-05-2023 15:02,gwq_chemical_parameter_manual_cgwb_pb_1961_2025,4915


In [26]:
top_coordinate_flags = coordinate_flags.head(15)

top_keys = set(
    tuple(row)
    for row in top_coordinate_flags[
        station_keys
    ].itertuples(index=False, name=None)
)

coordinate_detail = coordinate_df.copy()

coordinate_detail["_parsed_date"] = combined_df["_parsed_date"]
coordinate_detail["_date_only"] = combined_df["_date_only"]

coordinate_detail = coordinate_detail.dropna(
    subset=["_lat_num", "_lon_num"]
)

coordinate_detail["_station_key"] = list(
    zip(
        coordinate_detail["Station"],
        coordinate_detail["State"],
        coordinate_detail["District"]
    )
)

coordinate_detail = coordinate_detail[
    coordinate_detail["_station_key"].isin(top_keys)
]

coordinate_pair_summary = (
    coordinate_detail.groupby(
        station_keys + ["_lat_num", "_lon_num"],
        dropna=False
    )
    .agg(
        Records=("Station", "size"),
        Unique_Dates=("_date_only", "nunique"),
        First_Sampled=("_parsed_date", "min"),
        Last_Sampled=("_parsed_date", "max")
    )
    .reset_index()
)

print("COORDINATE CLUSTERS FOR THE TOP 15 FLAGGED IDENTITIES")
display(
    coordinate_pair_summary.sort_values(
        ["Station", "Records"],
        ascending=[True, False]
    )
)

COORDINATE CLUSTERS FOR THE TOP 15 FLAGGED IDENTITIES


,Station,State,District,_lat_num,_lon_num,Records,Unique_Dates,First_Sampled,Last_Sampled
1,Alampur,Andhra Pradesh,ANANTAPUR,14.191667,78.163889,19,19,2000-05-25 00:00:00,2018-05-25 00:00:00
2,Alampur,Andhra Pradesh,ANANTAPUR,26.029444,78.797222,18,18,2000-05-25 00:00:00,2022-05-31 11:00:00
0,Alampur,Andhra Pradesh,ANANTAPUR,14.189722,78.180556,2,2,2021-05-30 08:00:00,2022-05-20 18:29:00
4,Bantwaram,Telangana,RANGA REDDY,17.493333,78.340278,7,7,2013-05-25 00:00:00,2020-05-30 08:00:00
3,Bantwaram,Telangana,RANGA REDDY,17.446944,77.707778,1,1,2002-05-25 00:00:00,2002-05-25 00:00:00
5,Devri,Chhattisgarh,RAIPUR,20.882778,81.955833,11,11,2008-05-25 00:00:00,2023-12-01 11:12:00
6,Devri,Chhattisgarh,RAIPUR,21.466667,81.683333,4,4,2020-06-22 14:34:00,2023-12-01 10:29:00
8,Hazipur,Punjab,HOSHIARPUR,31.972222,75.750000,16,16,2000-05-25 00:00:00,2023-05-05 11:37:00
7,Hazipur,Punjab,HOSHIARPUR,31.266667,76.152778,6,6,2009-05-25 00:00:00,2018-05-25 00:00:00
10,Helencha P.s.,West Bengal,NORTH 24 PARGANAS,23.190000,88.860000,3,3,2002-05-25 00:00:00,2016-05-25 00:00:00


In [27]:
admin_df = combined_df.copy()

for column in ["State LGD Code", "District LGD Code"]:
    admin_df[column] = (
        admin_df[column].astype(str).str.strip()
    )

state_code_check = (
    admin_df.groupby("State LGD Code")
    .agg(
        State_Count=("State", "nunique"),
        State_Names=("State", lambda x: sorted(set(x))),
        Records=("State", "size")
    )
    .reset_index()
)

district_code_check = (
    admin_df.groupby("District LGD Code")
    .agg(
        District_Count=("District", "nunique"),
        District_Names=("District", lambda x: sorted(set(x))),
        State_Count=("State", "nunique"),
        State_Names=("State", lambda x: sorted(set(x))),
        Records=("District", "size")
    )
    .reset_index()
)

print("STATE CODE CONFLICTS")
display(
    state_code_check[state_code_check["State_Count"] > 1]
)

print("\nALL DISTRICT CODE CONFLICTS")

district_conflicts = district_code_check[
    (district_code_check["District_Count"] > 1)
    | (district_code_check["State_Count"] > 1)
]

display(district_conflicts)

print("\nIMPORTANT DISTRICT CODE MAPPINGS")

important_codes = ["441", "483", "509", "9999"]

display(
    admin_df[
        admin_df["District LGD Code"].isin(important_codes)
    ]
    .groupby(
        ["District LGD Code", "State LGD Code",
         "State", "District"]
    )
    .size()
    .reset_index(name="Records")
    .sort_values("District LGD Code")
)

print("\nTOTAL RECORDS USING DISTRICT CODE 9999:",
      (admin_df["District LGD Code"] == "9999").sum())

STATE CODE CONFLICTS


,State LGD Code,State_Count,State_Names,Records



ALL DISTRICT CODE CONFLICTS


,District LGD Code,District_Count,District_Names,State_Count,State_Names,Records
5,104,2,"[JALORE, Jalore]",1,[Rajasthan],181
23,120,2,"[Allahabad, PRAYGRAJ]",1,[Uttar Pradesh],178
86,179,2,"[Bhadohi, SANT RAVIDAS NAGAR]",1,[Uttar Pradesh],49
193,310,2,"[DAKSHIN DINAJPUR, South Dinajpur]",1,[West Bengal],134
194,311,2,"[NORTH DINAJPUR, UTTAR DINAJPUR]",1,[West Bengal],200
210,327,2,"[EAST SINGHBHUM (Jamshedpur), PURBI SINGHBHUM]",1,[Jharkhand],94
225,343,2,"[PASHCHIMI SINGHBHUM, WEST SINGHBHUM (Chaibasa)]",1,[Jharkhand],150
333,441,2,"[Banaskantha, DAMAN]",1,[Gujarat],320
374,483,2,"[MUMBAI CITY, Mumbai Suburban]",1,[Maharashtra],185
403,509,2,"[KHAMMAM, Khammam]",2,"[Andhra Pradesh, Telangana]",730



IMPORTANT DISTRICT CODE MAPPINGS


,District LGD Code,State LGD Code,State,District,Records
0,441,24,Gujarat,Banaskantha,250
1,441,24,Gujarat,DAMAN,70
2,483,27,Maharashtra,MUMBAI CITY,76
3,483,27,Maharashtra,Mumbai Suburban,109
4,509,28,Andhra Pradesh,Khammam,15
5,509,36,Telangana,KHAMMAM,715
6,9999,19,West Bengal,Calcutta,44
7,9999,19,West Bengal,KOLKATA,15
8,9999,34,Puducherry,Yanam,2



TOTAL RECORDS USING DISTRICT CODE 9999: 61



# PHASE 2 — DATASET AUDIT REPORT
## Borewell Sentinel India

**Phase:** 2 — Dataset Audit  
**Status:** COMPLETED  
**Source:** CGWB Groundwater Quality Chemical Monitoring Dataset (1961–2025 collection)

---

## 1. Dataset Overview

The project collected 31 state/UT-level CSV files from the original CGWB dataset archive.

| Metric | Verified Result |
|---|---:|
| Source CSV files | 31 |
| Total records | 174,635 |
| Original columns per file | 46 |
| States and UTs represented | 31 |
| Distinct district labels | 596 |
| Overall observation date range | 25 May 2000 – 8 May 2025 |
| Exact duplicate records | 0 |

All 31 datasets contain exactly 46 columns, with identical column names and column ordering.

The original files were successfully extracted and preserved.

## 2. Dataset Schema and Data Types

The 46 original columns comprise:

- Serial and monitoring station identifiers
- Agency and administrative information
- Geographic information
- Sampling date
- Groundwater chemical measurements

All columns were initially loaded as strings to preserve the original source values during the audit.

A combined working DataFrame was created containing the original 46 columns and additional audit fields, including:

- `_source_file`
- `_parsed_date`
- `_date_only`
- `_year`
- `_source_row`

These fields were added for analysis and do not replace original source fields.

## 3. Missing-Value Audit

The audit identified:

- 3,032,074 blank cells.
- 1,056,236 candidate placeholder cells.
- No detected candidate missing-value markers in the chemical measurement columns.

### Completely unavailable chemical parameters

The following parameters are blank across all 174,635 records:

| Parameter | Missing Records |
|---|---:|
| Nitrate N | 174,635 |
| Fluoride | 174,635 |
| Potassium | 174,635 |

These parameters cannot currently serve as supervised prediction targets using this dataset alone.

Other parameters, particularly trace metals, have very limited availability.

### Geographic and administrative fields

- Latitude and Longitude each have 152 blank values.
- River, Basin, Tributary, Subtributary, SubSubtributary and Local River contain `-` placeholders throughout.
- Block, Village and Tehsil also contain some `-` placeholders.

Blank values and explicit placeholders will be handled separately during cleaning.

## 4. Numeric Data-Quality Audit

Only six nonblank values failed numeric conversion.

| Column | Invalid Values |
|---|---:|
| pH | 4 |
| Total Alkalinity | 1 |
| Iron | 1 |
| All other numeric columns | 0 |

All six malformed entries originated from the Tamil Nadu dataset.

The affected measurements included concatenated-looking values such as `7.147.44`, `290.2575.4` and `0.030.05`.

These values will be preserved in the raw dataset and flagged as invalid in the processed dataset unless their correct values can be independently established.

## 5. Date and Temporal Coverage Audit

All 174,635 records have successfully parsed dates.

- Invalid dates: 0
- Earliest observation: 25 May 2000
- Latest observation: 8 May 2025

Temporal coverage is uneven.

Annual record counts:

| Period | Records / Observation |
|---|---:|
| 2000 | 6,645 |
| 2005 | 6,215 |
| 2010 | 6,967 |
| 2015 | 7,404 |
| 2016 | 10,650 |
| 2018 | 10,106 |
| 2019 | 8,111 |
| 2020 | 8,109 |
| 2021 | 5,994 |
| 2022 | 5,264 |
| 2023 | 2,522 |
| 2024 | 15 |
| 2025 | 1 |

The dataset has substantial historical coverage, but very limited nationwide coverage in 2024 and 2025.

Consequently, temporal predictions must account for the uneven sampling history and must not assume continuous nationwide monitoring.

## 6. Station Identity Audit

A provisional station identity was defined using:

`Station + State + District`

This produced 23,856 provisional station identities.

| Metric | Count |
|---|---:|
| Total provisional station identities | 23,856 |
| Identities with 1 record | 3,641 |
| Identities with 2+ records | 20,215 |
| Identities with 5+ records | 13,768 |
| Identities with 10+ records | 7,036 |

Station histories may support temporal analysis, but station names alone are insufficient to establish physical location identity.

Additional location-level validation is required.

### Station-name inconsistencies

- 33 station entries contain leading or trailing whitespace.
- Differences in capitalization and spelling occur.
- 692 station-name entries are associated with multiple states or districts.

Station names will not be used alone as unique identifiers.

## 7. Duplicate and Repeated Observation Audit

The audit found:

| Metric | Count |
|---|---:|
| Exact duplicate rows beyond the first occurrence | 0 |
| Station-date groups with multiple records | 280 |
| Records in those repeated groups | 565 |
| Excess observations beyond one per group | 285 |

Repeated station-date observations are not automatically duplicates.

Investigation showed that some records with the same station name, state, district and date have different geographic coordinates and chemical measurements.

Such observations must be retained unless independent evidence confirms duplication.

## 8. Geographic Coordinate Audit

| Metric | Result |
|---|---:|
| Records with both coordinates available | 174,483 |
| Records with both coordinates missing | 152 |
| Records with only one coordinate missing | 0 |
| Populated coordinates outside valid world ranges | 0 |
| Populated coordinates outside broad India screening envelope | 0 |

Observed coordinate ranges:

- Latitude: 8.08333333 to 34.65027778
- Longitude: 68.53333333 to 96.13333333

Although the overall coordinates pass the broad range checks, multiple provisional station identities have significantly different coordinate clusters.

Examples include Alampur, Madhudi, Suisa, Devri and Hazipur.

Alampur has two widely separated geographic clusters under the same provisional station identity. These records require location-level separation during integration.

Coordinate changes will not be treated as errors automatically. Original coordinates will be retained and investigated.

## 9. Administrative and Identifier Audit

### Serial numbers

- All 31 files contain unique serial numbers within their respective files.
- No blank serial numbers were detected.
- Serial numbers are not assumed to be globally unique across all files.

### Agency

All 174,635 records are attributed to CGWB.

### Administrative code inconsistencies

Several district LGD codes map to multiple district labels or states.

Important examples:

| District LGD Code | Observed Inconsistency |
|---|---|
| 104 | Jalore / JALORE |
| 120 | Allahabad / Prayagraj |
| 179 | Bhadohi / Sant Ravidas Nagar |
| 310 | Dakshin Dinajpur / South Dinajpur |
| 311 | North Dinajpur / Uttar Dinajpur |
| 327 | East Singhbhum / Purbi Singhbhum |
| 343 | Pashchimi Singhbhum / West Singhbhum |
| 441 | Banaskantha / Daman |
| 483 | Mumbai City / Mumbai Suburban |
| 509 | Khammam in Andhra Pradesh / Telangana |
| 566 | Thrissur / Trichur |
| 593 | Thiruvannamalai spelling variations |
| 704 | Barddhaman / Paschim Bardhaman |
| 9999 | Calcutta / Kolkata / Yanam |

District code `9999` occurs in 61 records.

Original administrative names and codes must be preserved. Any standardized administrative mapping will be maintained separately and should be validated before use.

## 10. Chemical Measurement Plausibility Audit

The numeric audit identified negative values, zeros, extreme measurements and suspicious pH readings.

### Suspicious pH values

Among successfully parsed numeric pH measurements:

- Four values were zero.
- Six values were outside the 0–14 screening range.

The six out-of-range values included 76, 88.12, 738, 796, 842 and 2,667.

These require quality flags and investigation.

### Negative chemical measurements

A total of 138 negative numeric measurements were detected across six parameters.

| Parameter | Negative Count |
|---|---:|
| Bicarbonate | 1 |
| Chloride | 1 |
| Sulphate | 2 |
| Silica | 116 |
| Magnesium | 10 |
| Iron | 8 |
| **Total** | **138** |

Some extremely small negative values are close to zero and may represent numerical precision effects. They should be assessed using a documented tolerance rather than treated identically to substantial negative concentrations.

### Extreme recorded values

Examples of recorded maximums requiring investigation:

| Parameter | Recorded Maximum |
|---|---:|
| pH | 2,667 |
| Electrical Conductivity | 57,006,760 μS/cm |
| Total Dissolved Solids | 2,353,897 mg/L |
| Bicarbonate | 817,830 mg/L |
| Arsenic | 345.455 mg/L |
| Calcium | 100,166 mg/L |
| Chloride | 309,472 mg/L |
| Total Hardness | 525,635 mg/L |
| Uranium | 2,429 mg/L |

These values are data-quality flags, not validated measurements of groundwater conditions.

No automatic outlier deletion, clipping or winsorization has been performed.

## 11. Key Risks Identified

The audit identified the following issues that must be considered during subsequent phases:

1. High and uneven missingness across chemical parameters.
2. Completely unavailable nitrate, fluoride and potassium measurements.
3. Six malformed numeric values.
4. Suspicious pH observations.
5. Negative chemical measurements.
6. Extreme recorded chemical values.
7. Inconsistent station names and formatting.
8. Multiple geographic locations associated with some provisional station identities.
9. Repeated station-date observations with differing measurements.
10. Inconsistent district names and administrative-code mappings.
11. Uneven sampling coverage, especially after 2022.

## 12. Cleaning and Integration Decisions

The following decisions will guide Phase 3:

- Preserve all original CSV files unchanged.
- Create a separate processed dataset.
- Convert measurement fields to numeric values only in the processed copy.
- Treat invalid numeric strings as missing in the processed copy while retaining audit flags.
- Standardize missing-value representations without confusing placeholders with valid zeros.
- Preserve legitimate zero measurements unless evidence establishes otherwise.
- Flag extreme values rather than automatically deleting them.
- Investigate negative measurements, with a separate tolerance policy for near-zero floating-point artifacts.
- Preserve original station labels and administrative codes.
- Construct more reliable station-location identities using geographic evidence.
- Retain repeated station-date records until their identity and measurement differences are understood.
- Exclude completely unavailable parameters from supervised modelling unless additional data becomes available.
- Document every cleaning operation so the process is reproducible.

## 13. Phase 2 Conclusion

The nationwide CGWB collection has been successfully loaded, structurally verified and audited.

It contains 174,635 records across 31 state/UT datasets, with substantial historical and geographic coverage.

However, the audit also identified important measurement-quality, missingness, administrative consistency and station-identity issues.

The dataset is suitable for proceeding to a controlled cleaning and integration stage, but it must not be treated as model-ready in its original form.

**Phase 2 — Dataset Audit: COMPLETED**

**Next Phase: Phase 3 — Data Cleaning & Integration**
